# Data Acquisition - NASA Turbofan Engine Dataset (C-MAPSS)
## Real-World Predictive Maintenance Data

**Dataset**: NASA's Commercial Modular Aero-Propulsion System Simulation (C-MAPSS)

**Why this dataset?**
- ✅ Real aircraft turbofan engine simulation data from NASA
- ✅ Run-to-failure sensor readings (perfect for predictive maintenance)
- ✅ 21 sensors measuring temperature, pressure, vibration, etc.
- ✅ Multiple operating conditions and fault modes
- ✅ Industry-standard dataset used in research and competitions
- ✅ Exactly matches OLPA's use case!

**Dataset Structure**:
- 4 sub-datasets (FD001, FD002, FD003, FD004) with increasing complexity
- ~100 engines per dataset
- Multiple sensor readings per operating cycle
- Training set: Run-to-failure data
- Test set: Partial runs (you predict RUL)

---

## Part 1: Download Dataset

**Source**: [NASA Prognostics Data Repository](https://data.nasa.gov/dataset/C-MAPSS-Aircraft-Engine-Simulator-Data/xaut-bemq)

**Direct Download Link**: https://data.nasa.gov/download/xaut-bemq/application%2Fzip

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import zipfile
import os
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ Libraries imported")

In [ ]:
# Create data directories
data_dir = Path('../data')
raw_dir = data_dir / 'raw' / 'nasa_cmapss'
raw_dir.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {raw_dir.absolute()}")

In [ ]:
# Download dataset
dataset_url = "https://data.nasa.gov/download/xaut-bemq/application%2Fzip"
zip_path = raw_dir / 'CMAPSSData.zip'

if not zip_path.exists():
    print(f"Downloading NASA C-MAPSS dataset...")
    print(f"URL: {dataset_url}")
    print("This may take a few minutes...\n")
    
    response = requests.get(dataset_url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    
    with open(zip_path, 'wb') as f:
        downloaded = 0
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total_size > 0:
                percent = (downloaded / total_size) * 100
                print(f"\rProgress: {percent:.1f}% ({downloaded:,} / {total_size:,} bytes)", end='')
    
    print(f"\n✓ Download complete: {zip_path}")
else:
    print(f"✓ Dataset already downloaded: {zip_path}")

In [ ]:
# Extract ZIP file
print("Extracting files...")

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(raw_dir)

print("✓ Extraction complete")

# List extracted files
print("\nExtracted files:")
for file in sorted(raw_dir.glob('*.txt')):
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"  {file.name:30s} - {size_mb:.2f} MB")

## Part 2: Understand the Data

**Dataset Files**:
- `train_FD00X.txt` - Training data (run-to-failure)
- `test_FD00X.txt` - Test data (partial runs)
- `RUL_FD00X.txt` - Remaining Useful Life labels for test data

**FD001-004 Differences**:
- **FD001**: Single operating condition, single fault mode (EASIEST)
- **FD002**: Multiple operating conditions, single fault mode
- **FD003**: Single operating condition, multiple fault modes
- **FD004**: Multiple operating conditions, multiple fault modes (HARDEST)

**Columns** (26 total, space-separated):
1. Engine ID
2. Time (in cycles)
3. Operational Setting 1
4. Operational Setting 2
5. Operational Setting 3
6-26. Sensor Measurements (21 sensors)

**Sensor Types**:
- Temperature sensors (T2, T24, T30, T50)
- Pressure sensors (P2, P15, P30, Ps30)
- Speed sensors (Nf, Nc)
- Fuel flow (Wf)
- And more...

In [ ]:
# Define column names (from NASA documentation)
sensor_columns = [
    'engine_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3',
    'sensor_1', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_5',
    'sensor_6', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_10',
    'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15',
    'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21'
]

# More descriptive sensor names (from NASA documentation)
sensor_names_mapping = {
    'sensor_2': 'T2_total_temp_LPC_outlet',
    'sensor_3': 'T24_total_temp_LPC_outlet',
    'sensor_4': 'T30_total_temp_HPC_outlet',
    'sensor_7': 'T50_total_temp_LPT_outlet',
    'sensor_8': 'P2_pressure_fan_inlet',
    'sensor_9': 'P15_total_pressure_bypass',
    'sensor_11': 'P30_total_pressure_HPC_outlet',
    'sensor_12': 'Nf_physical_fan_speed',
    'sensor_13': 'Nc_physical_core_speed',
    'sensor_14': 'epr_engine_pressure_ratio',
    'sensor_15': 'Ps30_static_pressure_HPC_outlet',
    'sensor_17': 'Wf_fuel_flow',
}

print("Column definitions loaded")
print(f"Total columns: {len(sensor_columns)}")

## Part 3: Load Training Data (FD001 - Simplest Dataset)

We'll start with FD001 (single operating condition, single fault mode) for learning.

In [ ]:
# Load FD001 training data
train_file = raw_dir / 'train_FD001.txt'

train_df = pd.read_csv(
    train_file,
    sep='\s+',  # Space-separated
    header=None,
    names=sensor_columns
)

print(f"✓ Loaded training data: {len(train_df):,} records")
print(f"✓ Number of engines: {train_df['engine_id'].nunique()}")
print(f"✓ Columns: {len(train_df.columns)}")
print("\nFirst few rows:")
display(train_df.head(10))

In [ ]:
# Dataset statistics
print("="*60)
print("DATASET STATISTICS")
print("="*60)

print(f"\nTotal Records: {len(train_df):,}")
print(f"Number of Engines: {train_df['engine_id'].nunique()}")

# Cycles per engine
cycles_per_engine = train_df.groupby('engine_id')['cycle'].max()
print(f"\nCycles per Engine (Life until failure):")
print(f"  Min:  {cycles_per_engine.min()}")
print(f"  Max:  {cycles_per_engine.max()}")
print(f"  Mean: {cycles_per_engine.mean():.1f}")
print(f"  Std:  {cycles_per_engine.std():.1f}")

print(f"\nData Types:")
print(train_df.dtypes.value_counts())

## Part 4: Create Target Variable (RUL - Remaining Useful Life)

**Key Concept**: For predictive maintenance, we need to predict how many cycles until failure.

In the training data:
- Each engine runs until failure
- RUL = (Max Cycle for that engine) - (Current Cycle)
- At failure: RUL = 0

In [ ]:
# Calculate RUL (Remaining Useful Life)
def add_rul(df):
    """Add RUL column based on max cycle per engine"""
    df = df.copy()
    
    # Get max cycle for each engine
    max_cycles = df.groupby('engine_id')['cycle'].max().reset_index()
    max_cycles.columns = ['engine_id', 'max_cycle']
    
    # Merge and calculate RUL
    df = df.merge(max_cycles, on='engine_id', how='left')
    df['RUL'] = df['max_cycle'] - df['cycle']
    df = df.drop('max_cycle', axis=1)
    
    return df

train_df = add_rul(train_df)

print("✓ RUL calculated")
print(f"\nRUL Statistics:")
print(train_df['RUL'].describe())

# Show example for one engine
print("\nExample: Engine 1 lifecycle")
display(train_df[train_df['engine_id'] == 1][['engine_id', 'cycle', 'RUL']].head(10))
print("...")
display(train_df[train_df['engine_id'] == 1][['engine_id', 'cycle', 'RUL']].tail(5))

## Part 5: Create Binary Target (Will Fail in 30 Days)

For OLPA's use case, we want to predict if failure will occur within 30 cycles (adjust as needed).

In [ ]:
# Create binary target: Will fail within 30 cycles?
PREDICTION_WINDOW = 30  # Adjust as needed (7, 14, 30 days)

train_df['will_fail_30_cycles'] = (train_df['RUL'] <= PREDICTION_WINDOW).astype(int)
train_df['failed'] = (train_df['RUL'] == 0).astype(int)

print(f"Binary target created: will_fail_30_cycles")
print(f"\nClass Distribution:")
print(train_df['will_fail_30_cycles'].value_counts())
print(f"\nClass Balance:")
print(train_df['will_fail_30_cycles'].value_counts(normalize=True))

print(f"\nActual Failures:")
print(f"  Total failure events: {train_df['failed'].sum()}")
print(f"  (One per engine)")

## Part 6: Visualize Degradation Patterns

In [ ]:
# Plot sensor degradation for sample engines
sample_engines = [1, 2, 3, 4, 5]

fig, axes = plt.subplots(len(sample_engines), 1, figsize=(16, 15))

# Key sensors to visualize
sensors_to_plot = ['sensor_4', 'sensor_11', 'sensor_7']  # Temperature and pressure

for idx, engine_id in enumerate(sample_engines):
    ax = axes[idx]
    
    engine_data = train_df[train_df['engine_id'] == engine_id].sort_values('cycle')
    max_cycle = engine_data['cycle'].max()
    
    # Plot sensors
    for sensor in sensors_to_plot:
        # Normalize to 0-1 for comparison
        normalized = (engine_data[sensor] - engine_data[sensor].min()) / \
                     (engine_data[sensor].max() - engine_data[sensor].min())
        ax.plot(engine_data['cycle'], normalized, label=sensor, linewidth=1.5)
    
    # Mark failure point
    ax.axvline(max_cycle, color='red', linestyle='--', linewidth=2, label='Failure', alpha=0.7)
    
    # Mark 30-cycle warning window
    if max_cycle >= PREDICTION_WINDOW:
        ax.axvspan(max_cycle - PREDICTION_WINDOW, max_cycle, 
                   alpha=0.2, color='orange', label=f'{PREDICTION_WINDOW}-Cycle Warning')
    
    ax.set_xlabel('Cycle')
    ax.set_ylabel('Normalized Sensor Value')
    ax.set_title(f'Engine {engine_id} - Degradation Pattern (Life: {max_cycle} cycles)')
    ax.legend(loc='best', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/nasa_degradation_patterns.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Degradation patterns visualized")

## Part 7: Sensor Analysis - Which Sensors are Most Predictive?

In [ ]:
# Calculate correlation with RUL and target variable
sensor_cols = [col for col in train_df.columns if col.startswith('sensor_')]

correlations = train_df[sensor_cols + ['RUL', 'will_fail_30_cycles']].corr()

# Extract correlations with target
rul_corr = correlations['RUL'][sensor_cols].abs().sort_values(ascending=False)
target_corr = correlations['will_fail_30_cycles'][sensor_cols].abs().sort_values(ascending=False)

print("Top 10 Sensors Correlated with RUL:")
print(rul_corr.head(10))

print("\nTop 10 Sensors Correlated with Failure Target:")
print(target_corr.head(10))

In [ ]:
# Visualize sensor importance
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RUL correlation
rul_corr.head(15).plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 Sensors - Correlation with RUL', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Absolute Correlation')
axes[0].set_ylabel('Sensor')

# Target correlation
target_corr.head(15).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title(f'Top 15 Sensors - Correlation with Failure ({PREDICTION_WINDOW} cycles)', 
                  fontsize=14, fontweight='bold')
axes[1].set_xlabel('Absolute Correlation')
axes[1].set_ylabel('Sensor')

plt.tight_layout()
plt.savefig('../docs/nasa_sensor_importance.png', dpi=300, bbox_inches='tight')
plt.show()

## Part 8: Data Quality Check

In [ ]:
print("="*60)
print("DATA QUALITY REPORT")
print("="*60)

print("\n1. Missing Values:")
missing = train_df.isnull().sum()
if missing.sum() == 0:
    print("  ✓ No missing values!")
else:
    print(missing[missing > 0])

print("\n2. Duplicates:")
duplicates = train_df.duplicated().sum()
print(f"  Duplicate rows: {duplicates}")

print("\n3. Constant Sensors (no variability):")
for col in sensor_cols:
    if train_df[col].std() < 0.001:
        print(f"  ⚠ {col} - std: {train_df[col].std():.6f} (consider removing)")

print("\n4. Data Types:")
print(train_df.dtypes.value_counts())

print("\n5. Value Ranges:")
print(train_df[sensor_cols].describe().T[['min', 'max', 'mean', 'std']].head(10))

## Part 9: Save Processed Data

In [ ]:
# Save to CSV for use in other notebooks
output_file = data_dir / 'raw' / 'sensor_data_nasa_fd001.csv'

train_df.to_csv(output_file, index=False)

print(f"✓ Saved processed data: {output_file}")
print(f"  Records: {len(train_df):,}")
print(f"  Size: {output_file.stat().st_size / (1024*1024):.2f} MB")
print(f"\nColumns saved:")
print(f"  {list(train_df.columns)}")

## Part 10: Load Other Datasets (FD002, FD003, FD004)

Optionally process the more complex datasets for advanced learning.

In [ ]:
# Function to load and process any FD dataset
def load_cmapss_dataset(dataset_name='FD001', prediction_window=30):
    """
    Load and process NASA C-MAPSS dataset
    
    Args:
        dataset_name: 'FD001', 'FD002', 'FD003', or 'FD004'
        prediction_window: Cycles to predict ahead
    
    Returns:
        DataFrame with RUL and target variables
    """
    train_file = raw_dir / f'train_{dataset_name}.txt'
    
    df = pd.read_csv(train_file, sep='\s+', header=None, names=sensor_columns)
    df = add_rul(df)
    df[f'will_fail_{prediction_window}_cycles'] = (df['RUL'] <= prediction_window).astype(int)
    df['failed'] = (df['RUL'] == 0).astype(int)
    
    print(f"✓ Loaded {dataset_name}: {len(df):,} records, {df['engine_id'].nunique()} engines")
    
    return df

# Example: Load all datasets
print("Loading all NASA C-MAPSS datasets...\n")

datasets = {}
for name in ['FD001', 'FD002', 'FD003', 'FD004']:
    datasets[name] = load_cmapss_dataset(name, prediction_window=PREDICTION_WINDOW)

print("\n✓ All datasets loaded")

In [ ]:
# Compare dataset complexities
print("="*60)
print("DATASET COMPARISON")
print("="*60)

comparison = pd.DataFrame({
    'Dataset': list(datasets.keys()),
    'Records': [len(df) for df in datasets.values()],
    'Engines': [df['engine_id'].nunique() for df in datasets.values()],
    'Avg_Cycles': [df.groupby('engine_id')['cycle'].max().mean() for df in datasets.values()],
    'Failure_Rate': [df[f'will_fail_{PREDICTION_WINDOW}_cycles'].mean() * 100 for df in datasets.values()]
})

display(comparison)

## Summary

**What you accomplished:**
1. ✅ Downloaded real NASA turbofan engine data
2. ✅ Loaded and understood the dataset structure
3. ✅ Created RUL (Remaining Useful Life) target variable
4. ✅ Created binary classification target (will_fail_30_cycles)
5. ✅ Visualized degradation patterns
6. ✅ Identified most predictive sensors
7. ✅ Assessed data quality
8. ✅ Saved processed data for next notebooks

**Next Steps:**
- Continue to `02_database_basics.ipynb` to load this real data into PostgreSQL
- Or `01_exploratory_data_analysis.ipynb` to do deeper EDA

**Dataset Choice for Learning:**
- **Start with FD001** (simplest, fastest to train)
- **Progress to FD002-004** once you understand the pipeline

---

**References:**
- NASA PCoE Datasets: https://data.nasa.gov/
- C-MAPSS Documentation: https://ntrs.nasa.gov/citations/20070034949